# Multi‑Modal RAG – Text + Image Understanding

**Author:** Ibrahim  
**Environment:** Google Colab / Python 3 (GPU required for BLIP)

## Overview
This notebook implements a **multi‑modal RAG system** that can answer questions about documents that contain both text and images (e.g., PDFs, reports, presentations). It uses:
- **OCR (Tesseract)** to extract text from images
- **BLIP (Bootstrapping Language-Image Pre-training)** to generate descriptive captions for images
- **FAISS** to index both text chunks and image captions
- **Groq LLM** to answer questions by retrieving relevant text or image descriptions

This project demonstrates RAG beyond pure text, a cutting‑edge skill for real‑world document understanding.

## Architecture
1. **Document Ingestion** – Upload PDFs. Pages are split into text and image chunks.
2. **Text Extraction** – PyPDFLoader extracts text; images are saved separately.
3. **Image Processing** – Each image is passed to BLIP for captioning.
4. **Indexing** – Text chunks and image captions are embedded and stored in FAISS.
5. **Retrieval** – User query is embedded; top‑k relevant chunks (text or image captions) are retrieved.
6. **Generation** – Retrieved context is sent to Groq LLM to produce an answer.

## Dataset
Any PDF with embedded images (e.g., annual reports, scientific papers, brochures).

## Requirements
- Groq API key
- PDFs with images (sample provided in code)

## Files Generated
- `faiss_multimodal_index/` – saved index
- `extracted_images/` – image files from PDF

---

**© 2026 Ibrahim – Multi‑modal RAG for text + images.**

### Install Dependencies

In [1]:
# RAG Project 5: Multi-Modal RAG (Text + Images)
# Author: Ibrahim

# Install only essential packages (avoid version conflicts)
!pip install -q --no-deps langchain langchain-community langchain-groq faiss-cpu sentence-transformers \
    langchain-huggingface langchain-text-splitters pypdf pillow chromadb \
    pdf2image pytesseract

# Install system dependencies
!apt-get install -qq -y poppler-utils tesseract-ocr > /dev/null

print("Dependencies installed. Proceeding with multi-modal RAG.")

Dependencies installed. Proceeding with multi-modal RAG.


### Imports

In [5]:
import os
import re
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# OCR and image processing
import pytesseract
from pdf2image import convert_from_path

# LangChain – using updated paths
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# BLIP for image captioning
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch

# Secure key input
from getpass import getpass

print("All libraries imported successfully.")

All libraries imported successfully.


### Set Up API Keys

In [6]:
GROQ_KEY = getpass("Enter your Groq API key: ")
os.environ["GROQ_API_KEY"] = GROQ_KEY
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)
print("LLM ready.")

Enter your Groq API key: ··········
LLM ready.


### Upload PDF and Extract Images + Text

In [7]:
# Upload PDF
print("Upload a PDF that contains images (e.g., a report or brochure).")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Convert PDF pages to images
images = convert_from_path(pdf_path, dpi=150)
print(f"Converted {len(images)} pages to images.")

# Extract text from each page using PyPDFLoader (for text chunks)
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"Loaded {len(pages)} text pages.")

# Directory to store extracted images
img_dir = "extracted_images"
os.makedirs(img_dir, exist_ok=True)
for i, img in enumerate(images):
    img.save(f"{img_dir}/page_{i+1}.png")

# OCR text extraction from images (fallback if PDF text extraction misses images)
ocr_texts = []
for i, img in enumerate(images):
    text = pytesseract.image_to_string(img)
    ocr_texts.append(text)
print("OCR text extracted from images.")

Upload a PDF that contains images (e.g., a report or brochure).


Saving Paper 3 ibrahim jcbi detection.pdf to Paper 3 ibrahim jcbi detection.pdf
Converted 11 pages to images.
Loaded 11 text pages.
OCR text extracted from images.


### Generate Image Captions using BLIP

In [9]:
# BLIP Image Captioning (Updated for reliability and performance)
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch
from tqdm import tqdm

# Force re-download of processor and model (in case of corrupt cache)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Move to GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"BLIP model loaded on {device}")

def caption_image(image_path):
    """Generate a caption for a single image."""
    try:
        image = Image.open(image_path).convert("RGB")
        # Process image and move tensors to the model's device
        inputs = processor(image, return_tensors="pt").to(device)
        # Generate caption with sensible defaults
        out = model.generate(
            **inputs,
            max_length=50,          # shorter captions for document figures
            num_beams=4,            # beam search for better quality
            temperature=0.9,
            do_sample=True          # slight randomness for varied captions
        )
        caption = processor.decode(out[0], skip_special_tokens=True)
        return caption
    except Exception as e:
        print(f"Error captioning {image_path}: {e}")
        return " [unable to caption] "

# Generate captions for all page images
captions = []
print("Generating captions for each page...")
for i in tqdm(range(len(images)), desc="Processing pages"):
    img_path = f"{img_dir}/page_{i+1}.png"
    cap = caption_image(img_path)
    captions.append(cap)
    # Print first 10 captions to see quality
    if i < 10:
        print(f"Page {i+1}: {cap}")

print(f"\n Generated {len(captions)} image captions.")

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

BLIP model loaded on cpu
Generating captions for each page...


Processing pages:   9%|▉         | 1/11 [00:09<01:38,  9.90s/it]

Page 1: a white paper with a black background and the title of the text


...

Page 9: a table with a bar graph and a bar graph on it


Processing pages:  91%|█████████ | 10/11 [01:21<00:09,  9.71s/it]

Page 10: a sample of a sample of a sample of a sample of a sample of a sample of a sample of a sample of a sample of a


Processing pages: 100%|██████████| 11/11 [01:30<00:00,  8.25s/it]


 Generated 11 image captions.


### Combine Text and Image Captions into Documents

In [10]:
# Create a list of document chunks: each chunk has content (text from page + caption)
documents = []
for i, page in enumerate(pages):
    text_content = page.page_content
    caption = captions[i] if i < len(captions) else ""
    combined = f"Text content: {text_content}\n\nImage description: {caption}\n\nOCR text: {ocr_texts[i]}"
    documents.append(combined)

# Split long text into chunks (optional)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_chunks = []
for doc in documents:
    chunks = text_splitter.split_text(doc)
    all_chunks.extend(chunks)

print(f"Total chunks (text + image captions): {len(all_chunks)}")

Total chunks (text + image captions): 72


### Create Multi‑Modal Vector Store

In [11]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(all_chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("FAISS index created with multi‑modal chunks.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index created with multi‑modal chunks.


### Build QA Chain

In [12]:
prompt_template = """
You are a multi‑modal document assistant. Answer the user's question based on the provided context, which includes text and image descriptions.

If the context contains an image description, treat it as a reliable source.
If the context is insufficient, say you don't know.

Context:
{context}

Question: {question}

Answer concisely:
"""
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)
print("QA chain ready.")

QA chain ready.


### Ask Questions

In [13]:
def ask_mm(query):
    result = qa_chain.invoke({"query": query})
    print(f"\n Question: {query}")
    print(f" Answer: {result['result']}")
    return result

# Example questions (customise based on your PDF)
sample_q = "What images or diagrams are present in the document?"
ask_mm(sample_q)


 Question: What images or diagrams are present in the document?
 Answer: The document contains the following images or diagrams:

1. A bar graph showing the number of different types of graphs.
2. A chart showing the number of different types of graphs (similar to the first bar graph).
3. A graph showing the Per-Class F1-Score for YOLOv8s on UATD.
4. A graph showing the YOLOv8s Training Loss Curves.
5. A sample image of a sample ( unclear description, likely an error in the text).
6. A graph showing Real Underwater Test Detections by YOLOv8s.
7. A graph showing Per-class detection showcase for all ten UATD obstacle categories.
8. A graph showing Class distribution across training and validation splits in the UATD dataset.


{'query': 'What images or diagrams are present in the document?',
 'result': 'The document contains the following images or diagrams:\n\n1. A bar graph showing the number of different types of graphs.\n2. A chart showing the number of different types of graphs (similar to the first bar graph).\n3. A graph showing the Per-Class F1-Score for YOLOv8s on UATD.\n4. A graph showing the YOLOv8s Training Loss Curves.\n5. A sample image of a sample ( unclear description, likely an error in the text).\n6. A graph showing Real Underwater Test Detections by YOLOv8s.\n7. A graph showing Per-class detection showcase for all ten UATD obstacle categories.\n8. A graph showing Class distribution across training and validation splits in the UATD dataset.',
 'source_documents': [Document(id='aea2603f-66bf-4480-b614-e3639091efa8', metadata={}, page_content='Image description: an image of a bar graph with a bar graph and a bar graph with a bar graph and a bar\n\nOCR text: Journal of Computing & Biomedical I

### Interactive Loop

In [16]:
print("\n" + "="*50)
print("Multi‑Modal RAG Ready. Ask about text or images in your PDF.")
print("Type 'exit' to quit.")
while True:
    q = input("\n Your question: ").strip()
    if q.lower() == "exit":
        break
    if not q:
        continue
    ask_mm(q)


Multi‑Modal RAG Ready. Ask about text or images in your PDF.
Type 'exit' to quit.

 Your question: What images or diagrams are present in the document?

 Question: What images or diagrams are present in the document?
 Answer: The document contains the following images or diagrams:

1. A bar graph showing the number of different types of graphs.
2. A chart showing the number of different types of graphs (similar to the first bar graph).
3. A graph showing the Per-Class F1-Score for YOLOv8s on UATD.
4. A graph showing the YOLOv8s Training Loss Curves.
5. A sample image of a sample ( unclear description, likely an error in the text).
6. A graph showing Real Underwater Test Detections by YOLOv8s.
7. A graph showing Per-class detection showcase for all ten UATD obstacle categories.
8. A graph showing Class distribution across training and validation splits in the UATD dataset.

 Your question: exit


### Final Summary

In [17]:
print("\n" + "="*50)
print("Multi‑Modal RAG - COMPLETED")
print("Author: Ibrahim")
print("="*50)
print(f" Processed PDF: {pdf_path}")
print(f" Extracted {len(images)} images.")
print(f" Generated captions for each image.")
print(f" Combined text + image descriptions into vector store.")
print(" You can now ask about both textual and visual content.")


Multi‑Modal RAG - COMPLETED
Author: Ibrahim
 Processed PDF: Paper 3 ibrahim jcbi detection.pdf
 Extracted 11 images.
 Generated captions for each image.
 Combined text + image descriptions into vector store.
 You can now ask about both textual and visual content.
